# Data centers
*Narrative prototype. Lets pretend that each markdown cell is our narrative that will come from scrolling in D3 and each visualization is how the scrollytelling will change the visualization.*

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd()
sys.path[:0] = [str(ROOT / ".pylibs"), str(ROOT / "scripts")]
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.graph_objects as go
CLEAN = ROOT / "data" / "clean"
dc = pd.read_csv(CLEAN / "datacenters.csv")
operating = dc[dc.status_group == "operating"]   # "Operating" + "Expanding"
planned = dc[dc.status_group == "pipeline"]      # proposed, approved, permitted, under construction

## Backstory: from Apollo to AI

**1950s–1960s.** Data centers emerged in the 1950s, supporting general computing, scientific research, and enterprise information systems. Initially concentrated within government agencies, universities, and large corporations, these facilities enabled breakthroughs that helped humanity achieve extraordinary technological milestones. In 1969, the Apollo 11 mission depended on IBM System/360 Model 75 mainframes in NASA's Real Time Computer Complex at Mission Control in Houston, Texas, alongside the Apollo Guidance Computer aboard the spacecraft, helping guide astronauts to the Moon.

**1970s–1980s.** Data centers expanded alongside the rise of mainframes and supercomputers, powering scientific research, weather forecasting, defense systems, and early digital business operations.

**1990s.** The growth of the internet transformed data centers into the backbone of global communications and e-commerce, supporting emerging platforms and online services.

**2000s.** Hyperscale computing arrived as companies such as Google, Amazon, and Microsoft built massive facilities capable of serving millions of users worldwide. Cloud computing, led by the 2006 launch of Amazon Web Services, shifted data centers from dedicated infrastructure to shared, on-demand platforms, dramatically increasing accessibility and scale.

**2012.** The resurgence of deep learning and advances in GPU computing sparked a new era of artificial intelligence. Data centers evolved beyond traditional computing environments, incorporating specialized accelerators, high-speed networking, and unprecedented processing power to train increasingly sophisticated AI models.

**2020s.** By 2023, generative AI systems such as large language models were being trained on clusters of tens of thousands of GPUs, driving the construction of purpose-built AI data centers. Today the largest AI campuses hold hundreds of thousands of GPUs, draw hundreds of megawatts to a gigawatt of power, and rely on advanced liquid cooling.

### Data centers' share of US electricity rose from 1.5% in 2006 to 4.4% in 2023, and it is projected to reach 6.7–12% by 2028.

In [2]:
elec = pd.DataFrame([(2000, 28.2, None), (2006, 61.4, 1.5), (2014, 58, None), (2016, 60, None), (2018, 76, 1.9), (2023, 176, 4.4)], columns=["year", "twh", "pct_us"])
eras = [(2000, 2006, "Internet era"), (2006, 2012, "Hyperscale & cloud"), (2012, 2020, "Deep learning"), (2020, 2028.4, "Generative AI")]
LINE, BAND = "#1f5fa8", "#1f5fa8"
fig = go.Figure()
for i, (a, b, name) in enumerate(eras):
    fig.add_vrect(x0=a, x1=b, fillcolor="#f2f2f2" if i % 2 == 0 else "white", line_width=0, layer="below")
    fig.add_annotation(x=(a + b) / 2, y=1.0, yref="paper", text=name, showarrow=False,
                       font=dict(size=12, color="#555"), yanchor="bottom")
hover = "%{x}: %{y} TWh<extra></extra>"

for seg, dash in [(elec[elec.year <= 2006], "solid"), (elec[elec.year.isin([2006, 2014])], "dash"), (elec[elec.year >= 2014], "solid")]:
    fig.add_trace(go.Scatter(x=seg.year, y=seg.twh, mode="lines", line=dict(color=LINE, width=2, dash=dash), hoverinfo="skip", showlegend=False))
    
fig.add_trace(go.Scatter(x=elec.year, y=elec.twh, mode="markers", marker=dict(size=8, color=LINE, line=dict(width=2, color="white")), hovertemplate=hover, showlegend=False))

# 2028 projection range
fig.add_trace(go.Scatter(x=[2023, 2028, 2028, 2023], y=[176, 580, 325, 176], mode="lines", fill="toself", fillcolor="rgba(31,95,168,0.15)", line=dict(width=0), hoverinfo="skip", showlegend=False))

fig.add_trace(go.Scatter(x=[2028, 2028], y=[325, 580], mode="markers", marker=dict(size=8, color="white", line=dict(width=2, color=LINE)), hovertemplate="2028 projection: %{y} TWh<extra></extra>", showlegend=False))

for x, y, t in [(2006, 61.4, "61 TWh<br>1.5% of US"), (2018, 76, "76 TWh<br>1.9%"), (2023, 176, "176 TWh<br>4.4%"), (2028, 580, "2028 projection<br>325–580 TWh<br>(6.7–12%)")]:
    fig.add_annotation(x=x, y=y, text=t, showarrow=False, xanchor="right", yanchor="bottom", xshift=-6, yshift=4, font=dict(size=11, color="#333"), align="right")
    
fig.add_vline(x=2022.9, line=dict(color="#999", width=1, dash="dot"))

fig.add_annotation(x=2022.9, y=420, text="ChatGPT<br>(Nov 2022)", showarrow=False, xanchor="right", xshift=-4, font=dict(size=11, color="#555"))

fig.update_layout(height=480, margin=dict(l=60, r=30, t=80, b=50), plot_bgcolor="white", title=dict(text="US data center electricity use, 2000–2023, and projection for 2028", x=0.02, y=0.97, font=dict(size=16)), xaxis=dict(range=[1999.5, 2028.8], dtick=4, showgrid=False, ticks="outside", linecolor="#999"), yaxis=dict(title="terawatt-hours per year", range=[0, 620], gridcolor="#eee", zeroline=False))

fig.add_annotation(x=0, y=-0.13, xref="paper", yref="paper", xanchor="left", showarrow=False, font=dict(size=10, color="#777"), text="Sources: EPA Report to Congress on Server and Data Center Energy Efficiency (2007); LBNL 2024 United States Data Center Energy Usage Report. Dashed: gap between reports.")
fig.show()

#### Meanwhile, each building got bigger. A 1950s air-defense computer drew a few megawatts. The typical data center operating today draws about 25. The typical one now being planned is about ten times that, and the largest AI campuses are approaching a gigawatt. Running at full power, a gigawatt is about 1.7 times the electricity used by every home in Memphis combined, and about two-thirds of what the entire city (homes, businesses, and industry) uses!

In [3]:
# City comparison quoted above: a 1 GW campus vs. Memphis's city utility (MLGW), EIA-861, 2025.
# Assumes the campus runs at full power all year (an upper bound; real draw is lower).
u = pd.read_csv(CLEAN / "eia861_utility_sales.csv")
mem = u[(u.utility_name == "City of Memphis - (TN)") & (u.year == 2025) & (u.part == "A")].iloc[0]
HOURS = 8760
campus_mw = 1000
homes_mw = mem.res_mwh / HOURS
city_mw = mem.total_mwh / HOURS
print(f"Memphis homes: {mem.res_customers:,.0f} customers, {mem.res_mwh / 1e6:.2f} TWh -> {homes_mw:,.0f} MW average")
print(f"Memphis total: {mem.total_mwh / 1e6:.2f} TWh -> {city_mw:,.0f} MW average")
print(f"1 GW campus = {campus_mw / homes_mw:.1f}x all Memphis homes, {campus_mw / city_mw:.0%} of the whole city")

Memphis homes: 385,015 customers, 5.27 TWh -> 602 MW average
Memphis total: 13.71 TWh -> 1,565 MW average
1 GW campus = 1.7x all Memphis homes, 64% of the whole city


In [4]:
# Visual B: how much power one facility draws, from the 1950s to today. Circle AREA is proportional to MW.
# Our-database medians use facilities with reported capacity (operating n=239, planned n=400).
# The two Memphis outlines (homes_mw, city_mw from the cell above) are placeholders for a city graphic in D3.
med_op = operating.mw.median(); med_pl = planned.mw.median()
scale = pd.DataFrame([
    ("SAGE air-defense computer<br>(1958)", 3, "AN/FSQ-7: up to 3 MW (Wikipedia)", "facility"),
    ("Typical operating<br>data center today", med_op, f"median of {operating.mw.notna().sum()} operating sites, our database", "facility"),
    ("xAI Colossus 1, Memphis<br>(2024)", 150, "grid supply upgraded toward ~150 MW (Wikipedia)", "facility"),
    ("Typical planned<br>data center", med_pl, f"median of {planned.mw.notna().sum()} planned sites, our database", "facility"),
    ("xAI Colossus sites,<br>Memphis–Southaven (2026)", 1000, "~1 GW combined by mid-2026 (Wikipedia, citing Epoch AI)", "facility"),
    ("All Memphis homes<br>(average draw, 2025)", homes_mw, "MLGW residential sales / 8,760 h (EIA-861, 2025)", "city"),
    ("All of Memphis<br>(homes, business, industry)", city_mw, "MLGW total sales / 8,760 h (EIA-861, 2025)", "city"),
], columns=["label", "mw", "source", "kind"])

# Layout in pixel units so circles sit on one baseline and never overlap.
W, H, TOP, BOTTOM = 1300, 440, 70, 10
MAX_D = 210                       # pixel diameter of the largest circle
MIN_SLOT, GAP, GROUP_GAP = 130, 20, 40
scale["d"] = MAX_D * np.sqrt(scale.mw / scale.mw.max())
slot = np.maximum(scale.d, MIN_SLOT) + GAP
left = slot.cumsum() - slot + np.where(scale.kind == "city", GROUP_GAP, 0)
scale["x"] = left + slot / 2
scale["y"] = scale.d / 2          # bottom of every circle on y = 0
sizeref = 2 * scale.mw.max() / MAX_D**2

fig = go.Figure()
styles = {"facility": dict(color="rgba(31,95,168,0.75)", line=dict(width=2, color="white")),
          "city": dict(color="rgba(120,120,120,0.06)", line=dict(width=1.5, color="#888"))}
for kind, s in scale.groupby("kind", sort=False):
    fig.add_trace(go.Scatter(x=s.x, y=s.y, mode="markers",
        marker=dict(size=s.mw, sizemode="area", sizeref=sizeref, **styles[kind]),
        customdata=s[["source"]], hovertemplate="%{marker.size:,.0f} MW<br>%{customdata[0]}<extra></extra>"))
for _, r in scale.iterrows():
    fig.add_annotation(x=r.x, y=0, yshift=-8, yanchor="top", showarrow=False, align="center",
                       text=f"<b>{r.mw:,.0f} MW</b><br>{r.label}", font=dict(size=12, color="#333" if r.kind == "facility" else "#666"))
city = scale[scale.kind == "city"]
fig.add_annotation(x=city.x.mean(), y=city.d.max() + 6, yanchor="bottom", showarrow=False,
                   text="<i>For scale: the city of Memphis</i>", font=dict(size=12, color="#666"))
plot_w, plot_h = W - 40, H - TOP - BOTTOM
fig.update_layout(width=W, height=H, margin=dict(l=20, r=20, t=TOP, b=BOTTOM), plot_bgcolor="white", showlegend=False,
    title=dict(text="Power drawn by a single facility, compared with a city (circle area proportional to megawatts)", x=0.02, font=dict(size=16)),
    xaxis=dict(visible=False, range=[0, plot_w]), yaxis=dict(visible=False, range=[-(plot_h - MAX_D - 30), MAX_D + 30]))
fig.write_html(ROOT / "jppolizzi" / "story_00b_facility_scale.html", include_plotlyjs=True, config={"displayModeBar": False})
fig.show()

#### From supporting the Apollo missions to powering today's AI boom, data centers have become critical infrastructure. 
#### But that infrastructure now needs as much electricity as a city, and often a great deal of water, and it is being built faster than ever.
#### Communities that are starting to ask what it costs them, and they are starting to push back against the expansion
*D3 notes: each era paragraph is one scroll step. Visual A draws its line era by era as the reader scrolls; Visual B grows one circle per step.*

#### We aim to understand if the underlying data supports datacenter expansion concerns and aim to understand which communities are most adversly effected by the build outs being planned

## 1. Datacenter build-out during the AI era (roughly 2020 to now)

The United States currently has **606** operating datacenters across **39** states. Some states have become highly concentrated with datacenter. Virginia has the highest concentration of datacenters containing 214 in the state. Thats more than the next two largest data center states combined, Georgia with 87 and Texas with 75. Another **975 facilities are planned** and if they are all built, the number of tracked sites would grow by **161%**.

In [5]:
# Numbers quoted above (recomputed so the text stays honest if the data changes)
by_state = operating.groupby("state").size().sort_values(ascending=False)
print(f"operating: {len(operating)} sites in {operating.state.nunique()} states")
print("top states:", by_state.head(3).to_dict())
print(f"planned: {len(planned)} sites -> +{len(planned) / len(operating) * 100:.0f}% if all built")

operating: 606 sites in 39 states
top states: {'VA': 214, 'GA': 87, 'TX': 75}
planned: 975 sites -> +161% if all built


In [6]:
# Top three states up close: Virginia (left), Texas and Georgia (right).
# Counties shaded by number of operating sites in 7 bins (light to dark blue), one dot per site.
import json
from plotly.subplots import make_subplots
counties = json.load(open(CLEAN / "counties_simplified.geojson"))
FIPS = {"VA": "51", "TX": "48", "GA": "13"}
op = operating.assign(geoid=operating.county_geoid.astype("Int64").astype(str).str.zfill(5))
county_n = op.groupby("geoid").size()
BIN_EDGES = [1, 2, 4, 10, 20, 50]   # bins: 0 | 1 | 2-3 | 4-9 | 10-19 | 20-49 | 50+
BIN_LABELS = ["0", "1", "2–3", "4–9", "10–19", "20–49", "50+"]
BIN_COLORS = ["#eef3fa", "#c6dbef", "#9ecae1", "#6baed6", "#3182bd", "#08519c", "#08306b"]
n_bins = len(BIN_COLORS)
binned_scale = [[edge, color] for i, color in enumerate(BIN_COLORS) for edge in (i / n_bins, (i + 1) / n_bins)]

fig = make_subplots(rows=2, cols=2, column_widths=[0.56, 0.44], horizontal_spacing=0.02, vertical_spacing=0.08,
                    specs=[[{"type": "geo", "rowspan": 2}, {"type": "geo"}], [None, {"type": "geo"}]],
                    subplot_titles=[f"{name}: {by_state[st]} operating sites" for st, name in [("VA", "Virginia"), ("TX", "Texas"), ("GA", "Georgia")]])
for (st, row, col), geo in zip([("VA", 1, 1), ("TX", 1, 2), ("GA", 2, 2)], ["geo", "geo2", "geo3"]):
    feats = [f for f in counties["features"] if f["properties"]["geoid"].startswith(FIPS[st])]
    ids = [f["properties"]["geoid"] for f in feats]
    names = [f["properties"]["name"] for f in feats]
    n = [county_n.get(g, 0) for g in ids]
    z = np.digitize(n, BIN_EDGES)   # bin index 0-6
    fig.add_trace(go.Choropleth(geojson={"type": "FeatureCollection", "features": feats}, featureidkey="properties.geoid",
        locations=ids, z=z, zmin=-0.5, zmax=n_bins - 0.5, colorscale=binned_scale,
        marker_line_color="white", marker_line_width=0.5, showscale=(st == "VA"), text=names, customdata=n,
        colorbar=dict(title="operating sites<br>per county", len=0.5, x=0.5, y=0.25,
                      tickvals=list(range(n_bins)), ticktext=BIN_LABELS),
        hovertemplate="%{text}: %{customdata} operating sites<extra></extra>"), row=row, col=col)
    s = op[op.state == st]
    fig.add_trace(go.Scattergeo(lon=s.lon, lat=s.lat, mode="markers", showlegend=False,
        marker=dict(size=5, color="#d94801", opacity=0.85, line=dict(width=0.3, color="white")),
        text=s.facility_name.fillna("(unnamed)") + "<br>" + s.city.fillna("") + ", " + st,
        hovertemplate="%{text}<extra></extra>"), row=row, col=col)
    fig.update_layout(**{geo: dict(fitbounds="locations", visible=False, projection=dict(type="mercator"))})
fig.update_layout(height=700, width=1300, margin=dict(l=10, r=10, t=80, b=10),
    title=dict(text="Three states hold more than half of today's operating data centers", x=0.02, font=dict(size=16)))
fig.write_html(ROOT / "jppolizzi" / "story_01a_top_states.html", include_plotlyjs=True, config={"displayModeBar": False})
fig.show()

In [7]:
# Map 1: current data center sites. States shaded by count of operating sites, one dot per site.
# Uses the same 7 bins and colors as the three-state map above (BIN_EDGES, BIN_LABELS, binned_scale).
STATES = ["AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "DC", "FL", "GA", "HI", "ID", "IL", "IN", "IA", "KS",
          "KY", "LA", "ME", "MD", "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ", "NM", "NY", "NC",
          "ND", "OH", "OK", "OR", "PA", "RI", "SC", "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY"]
counts = by_state.reindex(STATES, fill_value=0).rename("n").rename_axis("state").reset_index()
counts["bin"] = np.digitize(counts.n, BIN_EDGES)
fig = go.Figure()
fig.add_trace(go.Choropleth(locations=counts.state, z=counts.bin, locationmode="USA-states",
    colorscale=binned_scale, zmin=-0.5, zmax=n_bins - 0.5, customdata=counts.n,
    marker_line_color="white", marker_line_width=0.8,
    colorbar=dict(title="operating<br>data centers", len=0.55, tickvals=list(range(n_bins)), ticktext=BIN_LABELS),
    hovertemplate="%{location}: %{customdata} operating data centers<extra></extra>"))

fig.add_trace(go.Scattergeo(lon=operating.lon, lat=operating.lat, mode="markers", name="operating data center",
    marker=dict(size=7.5, color="#d94801", opacity=0.75, line=dict(width=0.85, color="white")),
    text=operating.facility_name.fillna("(unnamed)") + "<br>" + operating.city.fillna("") + ", " + operating.state,
    hovertemplate="%{text}<extra></extra>"))

fig.update_layout(height=600, margin=dict(l=0, r=0, t=50, b=0), legend=dict(x=0.02, y=0.06),
    title=dict(text=f"Where today's data centers are: {len(operating)} operating sites in {operating.state.nunique()} states", x=0.02, font=dict(size=16)),
    geo=dict(scope="usa", projection=dict(type="albers usa"), showlakes=False, bgcolor="white"))
fig.write_html(ROOT / "jppolizzi" / "story_01_current_datacenters.html", include_plotlyjs=True, config={"displayModeBar": False})
fig.show()

*Note for the D3 version: both maps share the same 7 bins, so a color means the same count on each; states and counties with no tracked site get the lightest color. The tracked database covers large and newsworthy facilities, so it undercounts small data centers, especially enterprise server rooms and smaller colocation sites.*

In [8]:
# Map 2: current + planned sites. States shaded by total sites if every planned site is built
# (same 7 bins and colors as Map 1). Filled orange = operating, open magenta ring = planned.
future = pd.concat([operating, planned])
future_counts = future.groupby("state").size().reindex(STATES, fill_value=0).rename("n").rename_axis("state").reset_index()
future_counts["bin"] = np.digitize(future_counts.n, BIN_EDGES)
future_counts["n_op"] = by_state.reindex(future_counts.state, fill_value=0).values
future_counts["n_pl"] = future_counts.n - future_counts.n_op

fig = go.Figure()

fig.add_trace(go.Choropleth(locations=future_counts.state, z=future_counts.bin, locationmode="USA-states",
    colorscale=binned_scale, zmin=-0.5, zmax=n_bins - 0.5, customdata=future_counts[["n", "n_op", "n_pl"]],
    marker_line_color="white", marker_line_width=0.8,
    colorbar=dict(title="total sites if<br>all are built", len=0.55, tickvals=list(range(n_bins)), ticktext=BIN_LABELS),
    hovertemplate="%{location}: %{customdata[0]} sites<br>%{customdata[1]} operating + %{customdata[2]} planned<extra></extra>"))

fig.add_trace(go.Scattergeo(lon=planned.lon, lat=planned.lat, mode="markers", name=f"planned ({len(planned)})",
    marker=dict(size=7.5, color="#4CAF50", opacity=0.85, line=dict(width=1.3, color="black")),
    text=planned.facility_name.fillna("(unnamed)") + "<br>" + planned.city.fillna("") + ", " + planned.state + "<br>" + planned.status,
    hovertemplate="%{text}<extra>planned</extra>"))

fig.add_trace(go.Scattergeo(lon=operating.lon, lat=operating.lat, mode="markers", name=f"operating ({len(operating)})",
    marker=dict(size=7.5, color="#d94801", opacity=0.85, line=dict(width=0.3, color="white")),
    text=operating.facility_name.fillna("(unnamed)") + "<br>" + operating.city.fillna("") + ", " + operating.state,
    hovertemplate="%{text}<extra>operating</extra>"))

fig.update_layout(height=600, margin=dict(l=0, r=0, t=50, b=0), legend=dict(x=0.02, y=0.06, traceorder="reversed"),
    title=dict(text=f"{len(operating)} operating sites, {len(planned)} more planned across {planned.state.nunique()} states", x=0.02, font=dict(size=16)),
    geo=dict(scope="usa", projection=dict(type="albers usa"), showlakes=False, bgcolor="white"))

fig.show()

## 2. Communities push back

As projects multiply, so does resistance. Across the country, cities, towns, townships, and counties have begun pausing new data centers with moratoriums. Our inventory records **1,036 adopted data center moratoriums in 47 states, and 946 are still in force**. County boards passed about a third of them, and city, town, township, and village councils passed the rest.

The wave is recent. Through most of 2025, communities adopted only a handful a month. In 2026 the pace exploded, with **170 adopted in June 2026** alone.

*Caveats: the inventory (Moratorium Nation, CC BY 4.0, compiled by Priyanka) was searched month by month only for May–July 2026, so part of the spike reflects search effort, and the recent months are likely incomplete. Most rows carry a "verify" flag from the dataset author.*

In [9]:
# Pushback numbers quoted above (recomputed so the text stays honest if the data changes)
m = pd.read_csv(CLEAN / "moratoria_local.csv", dtype={"county_geoid": str}, parse_dates=["date_enacted"])
adopted = m[m.adopted]   # drops proposals (pending), defeated proposals (rejected) and unknown status
print(f"adopted moratoria: {len(adopted)} in {adopted.state.nunique()} states; {adopted.in_force.sum()} still in force; {(m.enacted_status == 'pending').sum()} proposed")
print("passed by:", adopted.jurisdiction_type.value_counts(normalize=True).mul(100).round(0).to_dict())
monthly = adopted.set_index("date_enacted").resample("MS").size()
print("busiest month:", monthly.idxmax().strftime("%B %Y"), monthly.max())
print("top states:", adopted.state.value_counts().head(5).to_dict(), "| VA:", (adopted.state == "VA").sum(), "| TX:", (adopted.state == "TX").sum())

adopted moratoria: 1036 in 47 states; 946 still in force; 58 proposed
passed by: {'County': 35.0, 'City': 33.0, 'Township': 18.0, 'Town': 10.0, 'Village': 4.0, 'Other': 0.0, 'Tribal': 0.0, 'Utility-authority': 0.0}
busiest month: June 2026 170
top states: {'OH': 158, 'MI': 128, 'GA': 71, 'NC': 61, 'IA': 44} | VA: 6 | TX: 5


In [10]:
# Visual C: the pushback wave. Moratoriums adopted per month, 2025 onward (the 2018-2024 ones are left off so
# recent months are readable). The shaded band marks the months searched systematically, state by state.
PUSHBACK = "#b2182b"
wave = monthly[monthly.index >= "2025-01-01"]
fig = go.Figure(go.Bar(x=wave.index, y=wave.values, marker_color=PUSHBACK, marker_line_width=0,
    hovertemplate="%{x|%B %Y}: %{y} moratoriums adopted<extra></extra>"))
fig.add_vrect(x0="2026-04-16", x1="2026-07-16", fillcolor="#999", opacity=0.12, line_width=0,
              annotation_text="searched month by month", annotation_position="top left", annotation_font=dict(size=10, color="#777"))
fig.add_annotation(x=monthly.idxmax(), y=monthly.max(), text=f"<b>{monthly.max()}</b> in {monthly.idxmax():%b %Y}",
                   showarrow=False, yanchor="bottom", yshift=4, font=dict(size=12, color="#333"))
fig.add_annotation(x="2026-10-15", y=wave["2026-10-01":].max(), text="recent months<br>likely incomplete", showarrow=False,
                   yanchor="bottom", yshift=6, font=dict(size=10, color="#777"))
fig.update_layout(height=420, margin=dict(l=60, r=30, t=60, b=50), plot_bgcolor="white", bargap=0.15,
    title=dict(text="Data center moratoriums adopted per month", x=0.02, font=dict(size=16)),
    xaxis=dict(dtick="M3", tickformat="%b<br>%Y", showgrid=False, linecolor="#999", ticks="outside"),
    yaxis=dict(title="moratoriums adopted", gridcolor="#eee", zeroline=False, range=[0, monthly.max() * 1.25]))
fig.write_html(ROOT / "jppolizzi" / "story_03a_pushback_wave.html", include_plotlyjs=True, config={"displayModeBar": False})
fig.show()

Pushback isn't where data centers already are. Virginia hosts the most operating sites (214), yet it has only 6 moratoriums. The leaders are **Ohio (158), Michigan (128), and Georgia (71)**. Florida, Kansas, and Maine have no operating sites in our data but 68 moratoriums between them. Many of these are communities facing their first proposed data center, setting rules before it arrives.

In [11]:
# Map 3: where communities are pushing back. Each circle is the city, town, township, or county that adopted
# a moratorium (a reference point for the place, not the data center site).
# Filled = in force; ring = proposed. Grey dots are data center sites (operating + planned).
REASON_NAMES = {"regulatory_gap": "need time to write rules", "grid_energy": "grid and energy", "water": "water",
                "land_use_compatibility": "land use fit", "infrastructure_capacity": "roads, sewer, capacity",
                "specific_project": "a specific project", "noise": "noise", "environmental": "environment",
                "fire_safety": "fire safety", "property_values": "property values",
                "agricultural_preservation": "farmland", "legal_or_litigation": "legal", "other": "other"}
m["reasons"] = m[[f"reason_{k}" for k in REASON_NAMES]].apply(
    lambda r: ", ".join(REASON_NAMES[c[7:]] for c, v in r.items() if v) or "none coded", axis=1)
in_force = m[m.adopted & m.in_force]
proposed = m[m.enacted_status == "pending"]
sites = pd.concat([operating, planned])

def moratorium_trace(d, name, filled):
    return go.Scattergeo(lon=d.lon, lat=d.lat, mode="markers", name=name,
        marker=dict(size=7, color=PUSHBACK, opacity=0.8, symbol="circle" if filled else "circle-open",  # open symbols draw in color
                    line=dict(width=0.5 if filled else 1.5, color="white" if filled else PUSHBACK)),
        text=d.jurisdiction + ", " + d.state + "<br>" + d.enacted_status + ", " + d.date_enacted.dt.strftime("%b %Y").fillna("date unknown")
             + "<br>reasons: " + d.reasons,
        hovertemplate="%{text}<extra></extra>")

fig = go.Figure()
fig.add_trace(go.Choropleth(locations=STATES, z=[0] * len(STATES), locationmode="USA-states", showscale=False,
    colorscale=[[0, "#f2f2f2"], [1, "#f2f2f2"]], marker_line_color="white", marker_line_width=0.8, hoverinfo="skip"))
fig.add_trace(go.Scattergeo(lon=sites.lon, lat=sites.lat, mode="markers", name=f"data center site ({len(sites)})",
    marker=dict(size=3, color="#9a9a9a", opacity=0.5), hoverinfo="skip"))
fig.add_trace(moratorium_trace(proposed, f"moratorium proposed ({len(proposed)})", filled=False))
fig.add_trace(moratorium_trace(in_force, f"moratorium in force ({len(in_force)})", filled=True))
fig.update_layout(height=600, margin=dict(l=0, r=0, t=50, b=0), legend=dict(x=0.02, y=0.08, traceorder="reversed"),
    title=dict(text=f"Where communities are pushing back: {len(in_force)} data center moratoriums in force", x=0.02, font=dict(size=16)),
    geo=dict(scope="usa", projection=dict(type="albers usa"), showlakes=False, bgcolor="white"))
fig.write_html(ROOT / "jppolizzi" / "story_03b_pushback_map.html", include_plotlyjs=True, config={"displayModeBar": False})
fig.show()

What happens to projects that meet opposition? Only 3% of operating data centers have documented pushback. Among projects that stalled or were cancelled, about two-thirds did.

*Caveat: projects that drew opposition also drew news coverage, so their pushback is more likely to be recorded. Opposition clearly goes with stalled and cancelled projects, but this alone doesn't show it caused them.*

*Different source from the maps above. This chart uses the data center database's per-project flag (337 of 1,733 sites have documented local opposition to that specific project), not Priyanka's moratorium inventory. A moratorium is a town-wide pause on all data centers. This flag is opposition to one project. A site with no flag may still sit in a town with a moratorium, and the reverse.*

In [12]:
# Visual D: share of sites with documented community pushback, by project status.
# Source is the database's per-project pushback_documented flag, not moratoria_local.csv (see note above).
STATUS_ORDER = [("operating", "Operating"), ("pipeline", "Planned"), ("stalled", "Stalled"), ("cancelled", "Cancelled")]
share = (dc.assign(pb=dc.pushback_documented == True).groupby("status_group").pb.agg(["sum", "count"])
           .reindex([s for s, _ in STATUS_ORDER]))
share["pct"] = share["sum"] / share["count"] * 100
share["label"] = [name for _, name in STATUS_ORDER]
fig = go.Figure(go.Bar(y=share.label, x=share.pct, orientation="h", marker_color=PUSHBACK, marker_line_width=0,
    text=[f"<b>{p:.0f}%</b>  ({s:.0f} of {c:.0f})" for p, s, c in zip(share.pct, share["sum"], share["count"])],
    textposition="outside", cliponaxis=False, textfont=dict(size=12, color="#333"),
    hovertemplate="%{y}: %{x:.0f}% with documented opposition to the project<extra></extra>"))
fig.update_layout(height=330, margin=dict(l=100, r=60, t=60, b=40), plot_bgcolor="white", bargap=0.35,
    title=dict(text="Share of sites with documented opposition to that project, by project status", x=0.02, font=dict(size=16)),
    xaxis=dict(range=[0, 100], ticksuffix="%", gridcolor="#eee", zeroline=False),
    yaxis=dict(autorange="reversed", showgrid=False))
fig.write_html(ROOT / "jppolizzi" / "story_03c_pushback_outcomes.html", include_plotlyjs=True, config={"displayModeBar": False})
fig.show()

## So… what concerns are driving the pushback?

Every moratorium is an official action, and most state why it was passed. Nearly all of them (90%) say the community **needs time to write rules** for data centers. That is the mechanism, not the worry. Setting it aside, **72% name at least one specific harm**, and two lead the list: **the grid and energy, and water**, each cited by about a third.

*These are the reasons written into the official action, not a survey of residents. One moratorium can cite several reasons, so the shares add up to more than 100%.*

In [13]:
# Visual E: share of adopted moratoriums citing each harm. "Need time to write rules" (90%) and "other" are left off.
# The two concerns we test next (grid/energy and water) are colored; the rest are grey.
TESTED = {"grid_energy": "#1f5fa8", "water": "#1b9e9e"}
harms = [k for k in REASON_NAMES if k not in ("regulatory_gap", "other")]
cited = adopted[[f"reason_{k}" for k in harms]].mean().mul(100).set_axis(harms).sort_values()
fig = go.Figure(go.Bar(y=[REASON_NAMES[k].capitalize() for k in cited.index], x=cited.values, orientation="h",
    marker_color=[TESTED.get(k, "#bdbdbd") for k in cited.index], marker_line_width=0,
    text=[f"{v:.0f}%" for v in cited.values], textposition="outside", cliponaxis=False, textfont=dict(size=12, color="#333"),
    hovertemplate="%{y}: cited by %{x:.0f}% of moratoriums<extra></extra>"))
fig.update_layout(height=420, margin=dict(l=170, r=60, t=60, b=40), plot_bgcolor="white", bargap=0.3,
    title=dict(text=f"What moratoriums say they are about (share of {len(adopted):,} adopted)", x=0.02, font=dict(size=16)),
    xaxis=dict(range=[0, 45], ticksuffix="%", gridcolor="#eee", zeroline=False), yaxis=dict(showgrid=False))
fig.write_html(ROOT / "jppolizzi" / "story_04_concerns.html", include_plotlyjs=True, config={"displayModeBar": False})
fig.show()

Communities say they are worried about **the grid and their electric bills**, and about **water**. Are those worries backed by the data? We take them one at a time, starting with energy.

## 3. Is the grid under strain?

The grid is run by regional operators, each balancing supply and demand over its own territory. Every year, NERC (the North American Electric Reliability Corporation, the grid's reliability regulator) checks whether each region will have enough power plants to meet demand, and rates it **Normal, Elevated, or High risk** of running short.

Its latest assessment (January 2026) finds the outlook **"worsening"**, and names data centers as a main reason: *"Projections for resource and transmission growth lag what is needed to support new data centers and other large loads."*

Today, **65% of operating data centers (394 of 606) sit in regions rated Elevated risk**, including PJM, ERCOT (Texas), and SPP. By 2030, MISO, PJM, ERCOT, and parts of the West move to **High risk**, and **69% of operating sites and 69% of planned sites** are in those regions.

*Caveats: NERC rates whole regions, not individual towns or plants. The risk comes from fast demand growth (data centers are one driver) plus power plant retirements. Region boundaries here are approximate, built from the counties each utility serves.*

In [14]:
# Grid strain numbers quoted above: data center sites by the NERC risk rating of their grid region
nerc = pd.read_csv(CLEAN / "nerc_ltra_risk.csv")
county_ba = pd.read_csv(CLEAN / "county_ba.csv", dtype={"county_geoid": str})
NERC_YEARS = [2026, 2027, 2028, 2029, 2030]
site_risk = (pd.concat([operating, planned]).assign(county_geoid=lambda d: d.county_geoid.astype("Int64").astype(str).str.zfill(5))
             .merge(county_ba[["county_geoid", "ba_code"]], on="county_geoid", how="left").merge(nerc, on="ba_code", how="left"))
for y in (2026, 2030):
    print(y, pd.crosstab(site_risk[f"risk_{y}"].fillna("not assessed"), site_risk.status_group).to_dict())

2026 {'operating': {'Elevated': 394, 'Normal': 208, 'not assessed': 4}, 'pipeline': {'Elevated': 583, 'Normal': 387, 'not assessed': 5}}
2030 {'operating': {'Elevated': 52, 'High': 417, 'Normal': 133, 'not assessed': 4}, 'pipeline': {'Elevated': 91, 'High': 674, 'Normal': 205, 'not assessed': 5}}


In [15]:
# Map 4: grid regions colored by NERC risk rating, with a slider for 2026-2030. Dark dots are operating
# data centers, rings are planned ones. (Dots are dark here because orange and red now mean grid risk.)
import json
ba_geo = json.load(open(CLEAN / "ba_regions.geojson"))
RISK_LEVELS = ["Normal", "Elevated", "High"]
RISK_COLORS = {"Normal": "#dcdcdc", "Elevated": "#f4a259", "High": "#c0392b"}
regions = nerc[nerc.ba_code.isin([f["properties"]["ba_code"] for f in ba_geo["features"]])].copy()
area_sites = site_risk.groupby("nerc_area").status_group.value_counts().unstack(fill_value=0)
regions = regions.join(area_sites, on="nerc_area")
risk_scale = [[i / 3, RISK_COLORS[r]] for i, r in enumerate(RISK_LEVELS) for i in (i, i + 1)]  # stepped, z = 0, 1, 2

def region_trace(y):
    return go.Choropleth(geojson=ba_geo, featureidkey="properties.ba_code", locations=regions.ba_code,
        z=regions[f"risk_{y}"].map(RISK_LEVELS.index), zmin=-0.5, zmax=2.5, colorscale=risk_scale, showscale=False,
        marker_line_width=0, text=regions.nerc_area,
        customdata=regions[[f"risk_{y}", "operating", "pipeline"]].values,
        hovertemplate="<b>%{text}</b><br>%{customdata[0]} risk in " + str(y)
                      + "<br>%{customdata[1]} operating, %{customdata[2]} planned data centers<extra></extra>")

def title(y):
    n = (site_risk.status_group.eq("operating") & site_risk[f"risk_{y}"].isin(["Elevated", "High"])).sum()
    return f"Grid regions at risk of power shortfalls in {y}: {n} of {len(operating)} operating data centers are in them"

ops, pls = site_risk[site_risk.status_group == "operating"], site_risk[site_risk.status_group == "pipeline"]
fig = go.Figure([
    go.Choropleth(locations=STATES, z=[0] * len(STATES), locationmode="USA-states", showscale=False,
                  colorscale=[[0, "#f7f7f7"], [1, "#f7f7f7"]], marker_line_width=0, hoverinfo="skip"),
    region_trace(2026),
    go.Choropleth(locations=STATES, z=[0] * len(STATES), locationmode="USA-states", showscale=False,  # state lines on top
                  colorscale=[[0, "rgba(0,0,0,0)"], [1, "rgba(0,0,0,0)"]], marker_line_color="white", marker_line_width=0.8, hoverinfo="skip"),
    go.Scattergeo(lon=pls.lon, lat=pls.lat, mode="markers", name=f"planned data center ({len(pls)})", hoverinfo="skip",
                  marker=dict(size=4, symbol="circle-open", color="#333", opacity=0.45, line=dict(width=1))),
    go.Scattergeo(lon=ops.lon, lat=ops.lat, mode="markers", name=f"operating data center ({len(ops)})", hoverinfo="skip",
                  marker=dict(size=4, color="#222", opacity=0.8)),
] + [go.Scattergeo(lon=[None], lat=[None], mode="markers", name=f"{r} risk", marker=dict(size=12, symbol="square", color=RISK_COLORS[r]))
     for r in reversed(RISK_LEVELS)])
# Each slider step swaps only the region colors and hover text (trace 1) and the title, so the shapes are stored once
def year_step(y):
    t = region_trace(y)
    return dict(label=str(y), method="update",
                args=[dict(z=[list(t.z)], customdata=[t.customdata.tolist()], hovertemplate=[t.hovertemplate]), {"title.text": title(y)}, [1]])
fig.update_layout(height=640, margin=dict(l=0, r=0, t=50, b=40),
    title=dict(text=title(2026), x=0.02, font=dict(size=16)),
    legend=dict(x=0.02, y=0.12),
    geo=dict(scope="usa", projection=dict(type="albers usa"), showlakes=False, bgcolor="white"),
    sliders=[dict(active=0, x=0.3, len=0.6, y=0.02, currentvalue=dict(prefix="NERC rating for ", font=dict(size=13)),
                  steps=[year_step(y) for y in NERC_YEARS])])
fig.write_html(ROOT / "jppolizzi" / "story_05_grid_risk.html", include_plotlyjs=True, config={"displayModeBar": False})
fig.show()

*D3 notes: the slider becomes the scroll. Each scroll step advances a year and the regions turn from orange to red, so the reader watches the grid regions that hold most data centers move into high risk. Source: NERC 2025 Long-Term Reliability Assessment, Table 1 and Figure 1 (transcribed in `scripts/build_nerc_risk.py`).*

**Verdict on the grid: supported.** The grid's own reliability regulator says the risk of running short is rising and names data centers as a main reason. Most data centers already sit in regions rated Elevated, and by 2030 about two-thirds are in regions rated High.

*Caveats: this is a forecast of risk, not observed blackouts. Data centers are one driver alongside power plant retirements, and NERC rates whole regions, not the towns that host the buildings.*

## 4. Are data centers raising electric bills?

The most common worry is that data centers will raise everyone's electric bill. If that were happening state by state, the states with the most data centers should have seen the fastest price increases. So far, state by state, they haven't.

From 2019 to 2025, after inflation, the US average residential price rose **about 5%**. Virginia has by far the most data centers per resident, and its price was **flat (+0.6%)**. The biggest increases were in places with few data centers: DC, California, Maine, Maryland, and New York. Across states, the link runs slightly the *other* way (rank correlation −0.35).

But one group of states does stand out: those on the **PJM grid** (the Mid-Atlantic and Ohio Valley grid we just saw rated Elevated risk). Their prices rose a median **+9%**, against **−1%** everywhere else.

Coloring states by how their grid buys power shows a pattern. In states where one utility runs everything (the Southeast and most of the West), the median change was **−3%**. Regional markets without a capacity auction (Texas, SPP, California) came in at **−5%**, even with California's +30%. The other capacity auction markets (MISO, New York, New England) rose a median **+1%**. Only PJM stands out. The cost of new demand reaches bills fastest where an auction resets the price every year. Elsewhere it shows up later, if regulators let utilities pass on the cost of new plants.

**A timing warning:** the price data ends in 2025, but the PJM auction jump below only began reaching bills in June 2025, after a lag at most utilities. The two later auctions, which cost even more, are not in it at all. So this chart shows at most the start of the PJM effect, and the data center counts are a 2026 snapshot set against 2019–2025 prices.

*Analysis by Sanjana (`sgarimella/prototype_e_price_scatter.ipynb`). Data: EIA Form 861 residential revenue and sales, in 2025 dollars (BLS CPI-U). Caveats: the database has no build dates, so the x axis is how many data centers a state has now, not growth; a server room counts the same as a 1 GW campus; prices also move with fuel, wildfire, storm, and transmission costs; and each state is colored by the market serving most of its residential customers (EIA 861, 2025), so split states like Missouri (41% MISO) and Kentucky (39% PJM, shown as utility-run) are approximate.*

In [16]:
# Visual F: states, data centers per million residents vs. change in real residential price, 2019 to 2025 (after Sanjana's prototype E)
res = pd.read_csv(CLEAN / "eia861_state_residential.csv")
county_pop = pd.read_csv(CLEAN / "counties.csv", dtype={"geoid": str})
PRICE_BASE, PRICE_END = 2019, 2025
real_price = res.pivot(index="state", columns="year", values="res_price_cents_kwh_real2025")
sales_2025 = u[u.year == PRICE_END]
pjm_share = (sales_2025[sales_2025.ba_code == "PJM"].groupby("state").res_customers.sum()
             / sales_2025.groupby("state").res_customers.sum())
st_price = pd.DataFrame({
    "state_name": county_pop.drop_duplicates("state").set_index("state").state_name,
    "pop_m": county_pop.groupby("state").pop_2025.sum() / 1e6,
    "dc_live": pd.concat([operating, planned]).groupby("state").size(),
    "price_base": real_price[PRICE_BASE], "price_end": real_price[PRICE_END],
    "pjm_share": pjm_share}).loc[real_price.index]
st_price[["dc_live", "pjm_share"]] = st_price[["dc_live", "pjm_share"]].fillna(0)
st_price["dc_per_m"] = st_price.dc_live / st_price.pop_m
st_price["change_pct"] = (st_price.price_end / st_price.price_base - 1) * 100
st_price["pjm"] = st_price.pjm_share > 0.5   # most residential customers are served by PJM utilities
# How each state's grid buys power: the market type serving the most residential customers
MARKET = {"PJM": "PJM", "MISO": "capacity", "NYIS": "capacity", "ISNE": "capacity", "CISO": "regional", "ERCO": "regional", "SWPP": "regional"}
market_share = (sales_2025.assign(market=sales_2025.ba_code.map(MARKET).fillna("utility"))
                .groupby(["state", "market"]).res_customers.sum().unstack(fill_value=0))
st_price["market"] = market_share.idxmax(axis=1)
us_res = res.groupby("year")[["res_revenue_kusd", "res_mwh"]].sum()
cpi = pd.read_csv(CLEAN / "cpi_annual.csv").set_index("year").deflator_to_2025
us_price = us_res.res_revenue_kusd / us_res.res_mwh * 100 * us_res.index.map(cpi)
us_change = (us_price[PRICE_END] / us_price[PRICE_BASE] - 1) * 100
rho = st_price.dc_per_m.corr(st_price.change_pct, method="spearman")
print(f"US {us_change:+.1f}% | rank correlation {rho:.2f} | median change: PJM states {st_price[st_price.pjm].change_pct.median():+.1f}%, "
      f"others {st_price[~st_price.pjm].change_pct.median():+.1f}% | VA {st_price.loc['VA', 'change_pct']:+.1f}%")
print(st_price.groupby("market").change_pct.agg(["size", "median"]).round(1))

ZERO = 0.2   # where states with no data centers sit on the log axis
st_price["x"] = st_price.dc_per_m.where(st_price.dc_per_m > 0, ZERO)
LABELED = ["VA", "DC", "CA", "MD", "NJ", "ME", "NY", "PA", "OH", "GA", "TX", "IA", "WV", "ND", "WY"]
PJM_COLOR = "#e8833a"
MARKET_STYLE = [("utility", "#9a9a9a", "utility-run, no regional market (Southeast, most of the West)"),
                ("regional", "#1f5fa8", "regional market, no capacity auction (Texas, SPP, California)"),
                ("capacity", "#7b5ea7", "other capacity auctions (MISO, New York, New England)"),
                ("PJM", PJM_COLOR, "PJM capacity auction")]
fig = go.Figure()
for market, color, name in MARKET_STYLE:
    g = st_price[st_price.market == market]
    fig.add_trace(go.Scatter(x=g.x, y=g.change_pct, mode="markers+text", name=f"{name} ({len(g)})",
        text=[s if s in LABELED else "" for s in g.index], textposition="top center", textfont=dict(size=10, color="#555"),
        marker=dict(size=11, color=color, line=dict(width=1, color="white")),
        customdata=g.reset_index()[["state_name", "dc_live", "price_base", "price_end"]].values,
        hovertemplate="<b>%{customdata[0]}</b><br>%{customdata[1]} data centers (%{x:.1f} per million residents)"
                      "<br>%{customdata[2]:.1f} → %{customdata[3]:.1f} cents/kWh (2025 $)<br>%{y:+.1f}% after inflation<extra></extra>"))
fig.add_hline(y=us_change, line=dict(color="#777", dash="dash", width=1),
              annotation_text=f"US average {us_change:+.0f}%", annotation_position="top left", annotation_font=dict(size=11, color="#777"))
fig.add_hline(y=0, line=dict(color="#bbb", width=1))
fig.update_layout(height=520, margin=dict(l=70, r=30, t=60, b=60), plot_bgcolor="white",
    title=dict(text="More data centers did not mean faster price rises, but PJM states rose more", x=0.02, font=dict(size=16)),
    legend=dict(x=0.99, xanchor="right", y=0.99, title_text="how the state's grid buys power", bgcolor="rgba(255,255,255,0.85)"),
    xaxis=dict(type="log", title="operating and planned data centers per million residents (log scale)", range=[np.log10(0.15), np.log10(75)],
               tickvals=[ZERO, 0.5, 1, 2, 5, 10, 20, 50], ticktext=["0", "0.5", "1", "2", "5", "10", "20", "50"], gridcolor="#eee"),
    yaxis=dict(title=f"change in residential price, {PRICE_BASE}–{PRICE_END} (after inflation)", ticksuffix="%", gridcolor="#eee", zeroline=False))
fig.write_html(ROOT / "jppolizzi" / "story_06a_price_scatter.html", include_plotlyjs=True, config={"displayModeBar": False})
fig.show()

US +4.9% | rank correlation -0.35 | median change: PJM states +8.6%, others -0.9% | VA +0.6%
          size  median
market                
PJM          9     8.6
capacity    17     1.3
regional     6    -5.3
utility     19    -2.7


### Why would a whole grid region get more expensive?

PJM runs a yearly auction to buy enough power plant capacity for the future peak. Every utility in the region pays the result, so it reaches every household's bill. For three years the price sat at 50 dollars per MW-day or less. Then, in the auction for June 2025 to May 2026, it jumped 9.3x, from 28.92 to 269.92. 

The total cost rose from 2.2 billion to 14.7 billion, and the next two auctions pushed even higher, to 16.1 billion and 16.4 billion.

In [17]:
# Visual G: PJM capacity auction price (step) and a rough monthly bill add for a typical household (after Sanjana's prototype F)
bra = pd.read_csv(CLEAN / "pjm_bra_rto_prices.csv")
# Auction results are published in nominal dollars. Convert costs to 2025 dollars using the year each delivery year starts (June);
# 2026/27 and later are left as is.
deflator = cpi.copy()
deflator[2017] = 321.943 / 245.120   # 2025 CPI-U over the BLS 2017 annual average (2017 is not in cpi_annual.csv)
bra["cost_real_b"] = bra.cleared_cost_usd_billion * bra.delivery_year_start.clip(upper=2025).map(deflator)
pjm_load = pd.read_csv(CLEAN / "eia930_ba_annual.csv").query("ba == 'PJM' and complete_year").set_index("year")
bra["load_gwh"] = bra.delivery_year_start.clip(pjm_load.index.min(), pjm_load.index.max()).map(pjm_load.total_gwh_scaled)
bra["cents_per_kwh"] = bra.cost_real_b * 1e9 / (bra.load_gwh * 1e6) * 100   # cost spread over every kWh in PJM
pjm_res = u[(u.ba_code == "PJM") & (u.year == 2025)]
TYPICAL_KWH = pjm_res.res_mwh.sum() * 1000 / pjm_res.res_customers.sum() / 12
bra["usd_month"] = bra.cents_per_kwh * TYPICAL_KWH / 100
IMM_ACTUAL, IMM_NO_DC = 14.687, 5.355   # 2025/26 auction revenue, $B, with and without data center load (IMM Part G, Scenario 88)
jump = bra.set_index("delivery_year").loc["2025/26"]
dc_part = jump.usd_month * (1 - IMM_NO_DC / IMM_ACTUAL)
print(bra.set_index("delivery_year")[["cleared_cost_usd_billion", "cost_real_b", "usd_month"]].round(2).T.to_string())
print(f"typical PJM household {TYPICAL_KWH:.0f} kWh/month | 2025/26 add ${jump.usd_month:.2f}/month, of which data centers ~${dc_part:.2f}")

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.55, 0.45], vertical_spacing=0.12,
                    subplot_titles=("Capacity auction clearing price ($ per MW-day, as published)", f"Rough add to a typical household bill ({TYPICAL_KWH:.0f} kWh a month, 2025 dollars)"))
fig.add_trace(go.Scatter(x=bra.delivery_year, y=bra.rto_clearing_price_usd_per_mw_day, mode="lines+markers+text", line=dict(color=PJM_COLOR, width=3, shape="hvh"),
    marker=dict(size=7), text=[f"${v:,.0f}" for v in bra.rto_clearing_price_usd_per_mw_day], textposition="top center", textfont=dict(size=10, color="#333"),
    customdata=bra[["rpm_reserve_margin_pct", "cleared_cost_usd_billion"]].values, showlegend=False,
    hovertemplate="<b>%{x}</b><br>$%{y:,.2f} per MW-day<br>total cost $%{customdata[1]}B<br>reserve margin %{customdata[0]}%<extra></extra>"), 1, 1)
after = bra.delivery_year_start >= 2025
fig.add_trace(go.Bar(x=bra.delivery_year, y=bra.usd_month, marker_color=np.where(after, PJM_COLOR, "#c8c8c8"), marker_line_width=0, showlegend=False,
    text=[f"${v:.0f}" for v in bra.usd_month], textposition="outside", cliponaxis=False, textfont=dict(size=10, color="#333"),
    hovertemplate="%{x}: about $%{y:.2f} a month<extra></extra>"), 2, 1)
# The market monitor's split of the 2025/26 jump: the part due to data center load
fig.add_trace(go.Bar(x=["2025/26"], y=[dc_part], marker=dict(color="#a8481a", line_width=0),
    showlegend=False, hovertemplate=f"data center share (IMM): about ${dc_part:.0f} of ${jump.usd_month:.0f}<extra></extra>"), 2, 1)
fig.add_annotation(x="2025/26", y=dc_part / 2, text=f"~${dc_part:.0f}<br>data<br>centers", showarrow=False,
                   font=dict(size=10, color="white"), row=2, col=1)
fig.add_annotation(x="2025/26", y=jump.rto_clearing_price_usd_per_mw_day, text="9.3× jump", showarrow=True, ax=-60, ay=-30,
                   font=dict(size=12, color="#333"), arrowcolor="#555", row=1, col=1)
fig.update_layout(height=640, margin=dict(l=70, r=30, t=80, b=50), plot_bgcolor="white", barmode="overlay", bargap=0.3,
    title=dict(text="PJM capacity prices jumped ninefold; the market monitor attributes most of it to data centers", x=0.02, font=dict(size=16)))
fig.update_yaxes(gridcolor="#eee", zeroline=False)
fig.update_yaxes(range=[0, 400], row=1, col=1)
fig.update_yaxes(range=[0, 22], tickprefix="$", row=2, col=1)
fig.update_xaxes(showgrid=False)
fig.write_html(ROOT / "jppolizzi" / "story_06b_pjm_capacity.html", include_plotlyjs=True, config={"displayModeBar": False})
fig.show()

delivery_year             2017/18  2018/19  2019/20  2020/21  2021/22  2022/23  2023/24  2024/25  2025/26  2026/27  2027/28
cleared_cost_usd_billion     7.50    10.90     7.00     7.00     9.30     3.90     2.20     2.20     14.7    16.10    16.40
cost_real_b                  9.85    13.97     8.81     8.71    11.05     4.29     2.32     2.26     14.7    16.10    16.40
usd_month                   10.45    14.83     9.35     9.60    11.78     4.50     2.51     2.35     14.8    16.21    16.51
typical PJM household 849 kWh/month | 2025/26 add $14.80/month, of which data centers ~$9.40


Was that data centers? PJM's Independent Market Monitor (Monitoring Analytics, the market's official watchdog) tested this. It re-ran the 2025/26 auction with the same power plant offers, but took data center demand out of the forecast.

In [18]:
# Visual G1: the market monitor's re-run of the 2025/26 auction without data center load (IMM Part G, Table 7)
# The re-run is for 2025/26, so its numbers are already about 2025 dollars; earlier years use the converted costs from Visual G
prior = bra.set_index("delivery_year").cost_real_b["2024/25"]
imm = pd.DataFrame([
    ("2024/25 auction (actual)", prior, "#c8c8c8"),
    ("2025/26 re-run without any data center load", IMM_NO_DC_ALL := 5.355, "#f3c39b"),
    ("2025/26 re-run without new data center load", 6.944, "#eda36a"),
    ("2025/26 auction (actual)", 14.687, PJM_COLOR)], columns=["case", "cost_b", "color"])
fig = go.Figure(go.Bar(y=imm.case, x=imm.cost_b, orientation="h", marker_color=imm.color, marker_line_width=0,
    text=[f"${v:.1f}B" for v in imm.cost_b[:1]] + [f"${IMM_NO_DC_ALL:.1f}B   (+${IMM_NO_DC_ALL - prior:.1f}B from other causes)", "$6.9B",
          f"$14.7B   (+${14.687 - IMM_NO_DC_ALL:.1f}B from data centers)"], textposition="outside", cliponaxis=False, hovertemplate="%{y}: $%{x:.2f} billion<extra></extra>"))
early = bra[bra.delivery_year_start <= 2021].cost_real_b
fig.add_vrect(x0=early.min(), x1=early.max(), fillcolor="#eee", line_width=0, layer="below",
              annotation_text="2017/18–2021/22 range", annotation_position="top", annotation_font=dict(size=10, color="#777"))
fig.update_layout(height=340, margin=dict(l=320, r=60, t=60, b=50), plot_bgcolor="white", showlegend=False,
    title=dict(text="Without data centers, the 2025/26 auction would still have cost more than double the year before, but less than 2017–2021", x=0.02, font=dict(size=15)),
    xaxis=dict(title="total capacity auction cost ($ billion, 2025 dollars)", range=[0, 21], tickprefix="$", gridcolor="#eee"),
    yaxis=dict(autorange="reversed"))
fig.write_html(ROOT / "jppolizzi" / "story_06b_imm_rerun.html", include_plotlyjs=True, config={"displayModeBar": False})
fig.show()

Removing data centers takes the cost from **\$14.7 billion down to \$5.4 billion**. Removing only the *new* data centers in the forecast takes it to \$6.9 billion. The monitor concludes that data center load is *"the primary reason"* for the high prices, and that without it prices "would rise … but not jump suddenly to the maximum price."

The re-run does not say that data centers explain the whole jump. Even without them, the auction would have cost **more than double** 2024/25's record low (\$2.3B → \$5.4B). That is still less than any year from 2017 to 2021, which cost \$8.7–14.0B after inflation. Old plants retiring and rule changes pushed prices up too. And it is a model re-run that holds every bid fixed, not something anyone observed. The "without any data center load" case also removes data centers that already existed in 2024.

For a typical PJM household (about 850 kWh a month), capacity costs went from roughly **\$2 a month to \$15 a month**. By the monitor's numbers, about **\$3 of the increase would have happened anyway**, and about **\$9 is data center demand**. For perspective, \$15 a month is about what households paid in 2018/19 after inflation. The jump is large because it starts from the unusually cheap years of 2022–2025.

*Dollars: the bill and cost charts are in 2025 dollars (BLS CPI-U, by the year each delivery period starts). Clearing prices per MW-day and the auction totals quoted above are as published.*

**Verdict on bills: mixed, leaning supported.** State by state, more data centers did not mean faster price increases. But inside the PJM grid, data center demand drove most of a capacity cost that every household pays, about **\$9 a month** for a typical home in 2025/26. The cost is real. It is spread across a whole region rather than landing on the towns that host the buildings.

*Caveats: the monthly figure spreads the auction cost evenly over every kWh in PJM. Households use more power at peak times, so their real share is likely higher. Utilities pass these costs through with lags and hedges, which is also why the 2019–2025 state prices above catch only the start of this. Baltimore (BGE) and Dominion (Virginia) cleared higher, at \$466 and \$444 per MW-day. Part of the jump is a return from unusually cheap years, since before 2022/23 the same estimate ran \$9 to \$15 a month in 2025 dollars. Sources: PJM Base Residual Auction reports; Monitoring Analytics (PJM's Independent Market Monitor), "Analysis of the 2025/2026 RPM Base Residual Auction, Part G", June 3, 2025 (`data/raw/pjm_imm/`).*

*D3 notes: Sanjana's linked version (brush states on the scatter to draw their price lines) and usage slider (set your own kWh) carry over directly.*

## 5. Is there enough water?

About a third of moratoriums cite water. Many data centers cool their servers with water. The worry is sharpest where water is already scarce.

We can't measure how much water data centers use, because most don't report it. We *can* ask where they are being built. How many are landing in places that are already short of water?

The map shades each county by how much of it sits in a river basin under **High or Extremely high water stress**. That means people and industry already use more than 40% of the water that is available each year. Dots are data centers.

*Water stress is from WRI Aqueduct 4.0 (one baseline snapshot, not a trend). Built from Joel's prototypes A and B (`jppolizzi/prototype_a_map.ipynb`, `prototype_b_scorecard.ipynb`).*

In [19]:
# Map 5: county water stress (share of county area in High/Extremely high stress basins) with operating and planned data centers
water = pd.read_csv(CLEAN / "aqueduct_county.csv", dtype={"county_geoid": str})
county_names = pd.read_csv(CLEAN / "counties.csv", dtype={"geoid": str}).set_index("geoid")
water["name"] = water.county_geoid.map(county_names.county_name + ", " + county_names.state)
sites = (pd.concat([operating, planned]).query("coord_ok")
         .merge(pd.read_csv(CLEAN / "datacenters_basin.csv")[["facility_id", "stress_group"]], on="facility_id", how="left"))
sites["no_basin"] = sites.stress_group.isna() | sites.stress_group.eq("No data")   # Aqueduct has no value at the site (mostly metro Atlanta)
WATER_SCALE = [[0, "#f4f9f9"], [0.5, "#5fbcbc"], [1, "#0b5c5c"]]
# Only counties with some high-stress area are drawn; the rest show the light state fill underneath (keeps the file small)
stressed = water[water.share_high_or_extreme > 0.005]
stressed_ids = set(stressed.county_geoid)
stressed_geo = dict(counties, features=[f for f in counties["features"] if f["properties"]["geoid"] in stressed_ids])
fig = go.Figure([
    go.Choropleth(locations=STATES, z=[0] * len(STATES), locationmode="USA-states", showscale=False,
        colorscale=[[0, WATER_SCALE[0][1]], [1, WATER_SCALE[0][1]]], marker_line_width=0, hoverinfo="skip"),
    go.Choropleth(geojson=stressed_geo, featureidkey="properties.geoid", locations=stressed.county_geoid, z=stressed.share_high_or_extreme.round(2) * 100,
        zmin=0, zmax=100, colorscale=WATER_SCALE, marker_line_width=0, text=stressed.name,
        colorbar=dict(title="% of county in<br>high or extremely<br>high water stress", len=0.5, ticksuffix="%"),
        hovertemplate="%{text}<br>%{z:.0f}% in high or extremely high stress<extra></extra>"),
    go.Choropleth(locations=STATES, z=[0] * len(STATES), locationmode="USA-states", showscale=False,  # state lines on top
        colorscale=[[0, "rgba(0,0,0,0)"], [1, "rgba(0,0,0,0)"]], marker_line_color="white", marker_line_width=0.8, hoverinfo="skip")])
for status, name, marker in [
        ("pipeline", "planned data center", dict(size=4, symbol="circle-open", color="#333", opacity=0.45, line=dict(width=1))),
        ("operating", "operating data center", dict(size=4, color="#222", opacity=0.8))]:
    s = sites[(sites.status_group == status) & ~sites.no_basin]
    fig.add_trace(go.Scattergeo(lon=s.lon, lat=s.lat, mode="markers", name=f"{name} ({len(s)})", marker=marker,
        text=s.facility_name.fillna("(unnamed)") + "<br>" + s.city.fillna("") + ", " + s.state + "<br>basin stress: " + s.stress_group,
        hovertemplate="%{text}<extra></extra>"))
gap = sites[sites.no_basin]
fig.add_trace(go.Scattergeo(lon=gap.lon, lat=gap.lat, mode="markers", name=f"no water data at site ({len(gap)}, left out of the numbers)",
    marker=dict(size=5, symbol="x-thin", color="#d94801", line=dict(width=1.2, color="#d94801")),
    text=gap.facility_name.fillna("(unnamed)") + "<br>" + gap.city.fillna("") + ", " + gap.state, hovertemplate="%{text}<br>no Aqueduct data<extra></extra>"))
fig.update_layout(height=640, margin=dict(l=0, r=0, t=50, b=0),
    title=dict(text="Where water is already scarce, and where data centers are being built", x=0.02, font=dict(size=16)),
    legend=dict(x=0.02, y=0.1, bgcolor="rgba(255,255,255,0.8)"),
    geo=dict(scope="usa", projection=dict(type="albers usa"), showlakes=False, bgcolor="white"))
fig.write_html(ROOT / "jppolizzi" / "story_07a_water_map.html", include_plotlyjs=True, config={"displayModeBar": False})
fig.show()

Many are. **About a third of data centers sit in basins under High or Extremely high water stress: 34% of operating sites and 33% of planned ones, roughly 500 sites in all.** Most of them are in Virginia, Texas, Georgia, Illinois (around Chicago), and Arizona (around Phoenix). The planned build-out isn't moving away from those basins.

*Side note: about 36% of Americans also live in those basins. So data centers aren't seeking out dry places; they follow people and power lines. That doesn't ease the worry, though. A data center in a stressed basin draws on the same water as the people who live there.*

In [20]:
# Visual H: operating and planned data centers by the water stress of their river basin (after Joel's prototype B).
# Each facility is placed in its river basin from its coordinates. Bars are shares of sites with basin data, labelled with counts.
HIGH_STRESS = ["High", "Extremely high"]
STRESS_STYLE = {"Extremely high": "#0b5c5c", "High": "#2e8b8b", "Medium-high": "#8fd0d0",   # most stressed first, so the high-stress third reads from zero
                "Low-medium": "#cfe8e8", "Low": "#eef6f6", "Arid, low use": "#d9d9d9"}
STATUS_LABEL = {"operating": "operating", "pipeline": "planned"}
known = sites[~sites.no_basin]
counts = pd.crosstab(known.status_group, known.stress_group).reindex(index=list(STATUS_LABEL), columns=list(STRESS_STYLE), fill_value=0)
shares = counts.div(counts.sum(axis=1), axis=0) * 100
high = counts[HIGH_STRESS].sum(axis=1)
print(counts.assign(high_stress=high, high_share=(shares[HIGH_STRESS].sum(axis=1)).round(1)).to_string())
print("high-stress sites, total:", high.sum(), "| by state:", known[known.stress_group.isin(HIGH_STRESS)].state.value_counts().head(5).to_dict())
basin_base = pd.read_csv(CLEAN / "basin_population.csv").set_index("stress_group")
print("for reference, US population in high-stress basins: %.1f%%" % (basin_base.loc[HIGH_STRESS, "population_share"].sum() * 100))

ylabels = [f"{STATUS_LABEL[s]} ({counts.loc[s].sum()} sites)" for s in counts.index]
fig = go.Figure()
for group, color in STRESS_STYLE.items():
    fig.add_trace(go.Bar(y=ylabels, x=shares[group], orientation="h", name=group, marker_color=color, marker_line_width=0,
        text=[f"{n}" if p >= 4 else "" for n, p in zip(counts[group], shares[group])], textposition="inside", insidetextanchor="middle",
        textfont=dict(color="white" if group in HIGH_STRESS else "#0b3c3c"),
        customdata=counts[group], hovertemplate=f"%{{y}}<br>{group}: %{{customdata}} sites (%{{x:.0f}}%)<extra></extra>"))
for y, s in zip(ylabels, counts.index):   # bracket label over the high-stress part of each bar
    fig.add_annotation(x=shares.loc[s, HIGH_STRESS].sum(), y=y, yshift=26, xanchor="right", showarrow=False,
        text=f"<b>{shares.loc[s, HIGH_STRESS].sum():.0f}% high or extremely high ({high[s]})</b>", font=dict(color="#0b5c5c", size=12))
fig.update_layout(barmode="stack", height=320, margin=dict(l=170, r=40, t=60, b=80), plot_bgcolor="white", bargap=0.5,
    title=dict(text="About a third of data centers sit in basins already under high water stress", x=0.02, font=dict(size=16)),
    xaxis=dict(range=[0, 100], ticksuffix="%", gridcolor="#eee", title="share of sites, by water stress of their river basin (numbers are site counts)"),
    yaxis=dict(autorange="reversed"), legend=dict(orientation="h", traceorder="normal", x=0, y=-0.45, title="basin water stress: "))
fig.write_html(ROOT / "jppolizzi" / "story_07b_water_siting.html", include_plotlyjs=True, config={"displayModeBar": False})
fig.show()

stress_group  Extremely high  High  Medium-high  Low-medium  Low  Arid, low use  high_stress  high_share
status_group                                                                                            
operating                 31   166           98          66  211              2          197        34.3
pipeline                  93   215          242         130  255              6          308        32.7
high-stress sites, total: 505 | by state: {'VA': 150, 'TX': 117, 'GA': 46, 'IL': 29, 'AZ': 28}
for reference, US population in high-stress basins: 36.0%


**Verdict on water: can't tell.** Part of the worry is backed by the data: about a third of data centers, roughly 500 sites, sit in basins where people and industry already use more than 40% of the available water. But the real question is whether they are draining those basins, and that depends on how much water each facility uses and where it comes from. Those numbers are mostly unreported, and cooling type is known for only a few sites. Without them we can show *where* the risk is, not *whether* it is happening.

*Caveats: Aqueduct is one baseline snapshot (it shows scarcity, not change). Each data center is placed in its river basin from its coordinates. The county shading on the map summarizes the same basins by area. 66 sites (43 in Georgia, mostly metro Atlanta, and 13 in Virginia) fall where Aqueduct has no value and are left out of the shares, so Georgia is undercounted. A server room counts the same as a gigawatt campus. Some facilities use air cooling or recycled water and draw little.*

*D3 notes: on scroll, the map can dim every site outside a high-stress basin, leaving the ~500 that are, before the bar chart gives the count.*

## Sources and how they fit together

| Source | Used for | Unit | Time | Dollars |
|---|---|---|---|---|
| Data Centers Database (`datacenters.csv`) | every data center map and count; per-state counts in the price scatter; project opposition flag (Visual D) | facility (a server room counts the same as a campus) | current snapshot, no build dates | n/a |
| Moratorium Nation, compiled by Priyanka (`moratoria_local.csv`) | pushback timeline, map, concerns | local law (moratorium) | adopted through Nov 2026; searched month by month only May–July 2026 | n/a |
| EPA 2007 and LBNL 2024 reports | backstory electricity chart | national TWh | 2000–2023, projection 2028 | n/a |
| EIA Form 861 | residential prices; customers; utility → grid region (BA) for counties and states; typical household kWh | utility × state × year | 2019–2025 (prices); 2025 (grid assignments) | converted to 2025 \$ |
| EIA Form 930 | PJM annual load (spreads auction cost per kWh) | grid region × year | complete years, 2019–2025 | n/a |
| BLS CPI-U | inflation adjustment | year | 2017–2025 (2025 is 11 months) | n/a |
| NERC 2025 Long-Term Reliability Assessment (published January 2026) | grid risk map | NERC assessment area, matched to grid regions | 2026–2030 projection | n/a |
| PJM Base Residual Auction reports | capacity prices and total cost | PJM-wide, per delivery year (June–May) | 2017/18–2027/28 | prices as published; costs converted to 2025 \$ |
| PJM Independent Market Monitor, Part G (June 2025) | auction re-run without data centers | PJM-wide, MW of data center load in PJM's forecast | 2025/26 auction only | as published (≈ 2025 \$) |
| WRI Aqueduct 4.0 (`aqueduct_county.csv`, `datacenters_basin.csv`, `basin_population.csv`) | water stress map and site counts by stress level | river basin (facilities placed by coordinates; counties and population summarized by area overlap) | one baseline snapshot | n/a |

**Where they don't line up:**
- **Time.** Prices end in 2025, while the auction jump began reaching bills in mid-2025. Data center counts and NERC risk are current or forward-looking, so they are compared against past prices.
- **What "data center" means.** The database counts buildings, the market monitor counts megawatts of forecast load, and NERC speaks of "large loads." We only carry over the monitor's *ratio* (share of cost due to data centers), never its MW.
- **Pushback.** Section 2 uses town-wide moratoriums, except Visual D, which uses opposition to individual projects.
- **Water.** Data centers are placed in river basins, but the map shades counties. Both come from the same Aqueduct basins, and 66 sites with no basin value are left out of the shares.
- **Geography.** Counties and states are assigned to the grid region serving most of their residential customers (EIA-861, 2025). NERC areas are matched to those regions approximately (Wyoming, northern Nevada), and split states (Missouri, Kentucky) are approximate.